[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrenansantos-teaching/INFNET_Projeto_Bloco_IA_Aplicada_2026_2_alunos/blob/main/aula-02-colab/aula2_coleta_via_api.ipynb)

# Aula 2 — Coleta de dados via API do IBGE (Google Colab)
**Projeto de Bloco: IA Aplicada — INFNET**

Versão *notebook* do demo da Aula 2. Aqui rodamos a **coleta** e a **manipulação** em **Python puro** (`requests` + `json`, **sem pandas**). A **visualização interativa** fica no app **Streamlit** (ambiente local); no fim, um gráfico rápido com `matplotlib` só para conferir no notebook.

> A API do IBGE é **pública e gratuita** (sem chave).

In [ ]:
!pip install requests -q   # no Colab já vem instalado; garante a lib

## 1. Coletar as UFs (API de Localidades)
`GET` no *endpoint* → resposta em **JSON** → lista de dicionários.

In [ ]:
import requests

URL_ESTADOS = "https://servicodados.ibge.gov.br/api/v1/localidades/estados?orderBy=nome"
resp = requests.get(URL_ESTADOS, timeout=20)
resp.raise_for_status()                 # erro se status != 2xx
estados = resp.json()                   # JSON -> lista de dicts
por_id = {int(e["id"]): {"uf": e["sigla"], "estado": e["nome"], "regiao": e["regiao"]["nome"]}
          for e in estados}
print(len(por_id), "UFs. Exemplo:", por_id[35])

## 2. Coletar população por UF (API de Agregados) — JSON **aninhado**
A resposta é um JSON profundo; navegamos até as *séries* e montamos **uma linha (dict) por UF**.

In [ ]:
URL_POP = ("https://servicodados.ibge.gov.br/api/v3/agregados/6579"
           "/periodos/-1/variaveis/9324?localidades=N3[all]")
resp_pop = requests.get(URL_POP, timeout=20)
resp_pop.raise_for_status()
series = resp_pop.json()[0]["resultados"][0]["series"]

linhas = []
for s in series:
    id_uf = int(s["localidade"]["id"])            # id vem como TEXTO -> int
    valor = int(list(s["serie"].values())[0])     # {"2025": "46081801"} -> valor
    linha = dict(por_id[id_uf]); linha["populacao_2025"] = valor
    linhas.append(linha)

print(len(linhas), "linhas. Exemplo:", linhas[0])

## 3. Explorar em Python puro (len, set, sum)

In [ ]:
print("UFs:", len({l['uf'] for l in linhas}))
print("Regiões:", len({l['regiao'] for l in linhas}))
print("População do Brasil (soma):", f"{sum(l['populacao_2025'] for l in linhas):,}".replace(",", "."))
# top 5 mais populosas
for l in sorted(linhas, key=lambda x: -x["populacao_2025"])[:5]:
    print(l["uf"], l["estado"], f"{l['populacao_2025']:,}".replace(",", "."))

## 4. Filtrar por região (list comprehension)

In [ ]:
regiao = "Nordeste"
nordeste = [l for l in linhas if l["regiao"] == regiao]
print(f"{len(nordeste)} UFs no {regiao}:", [l["uf"] for l in nordeste])

## 5. Visualização rápida (matplotlib)
No **app Streamlit** usamos `st.bar_chart(dados, x='uf', y='populacao_2025')`. Aqui, só para conferir no notebook, um gráfico simples com `matplotlib` (já vem no Colab).

In [ ]:
import matplotlib.pyplot as plt

top = sorted(linhas, key=lambda l: -l["populacao_2025"])[:10]
plt.figure(figsize=(10, 4))
plt.bar([l["uf"] for l in top], [l["populacao_2025"] for l in top], color="#4F81BD")
plt.title("População por UF (top 10)")
plt.ylabel("População (2025)")
plt.show()

## Levar para o app Streamlit (local)
Este código de coleta vira `src/data_access.py` (isolado da interface). No app local:

```bash
pip install streamlit requests
streamlit run app.py
```

No Colab **não** rodamos o Streamlit; o app roda no seu ambiente local ou no **Streamlit Community Cloud**.